In [ ]:

# ============================================================
# PROJECT 5 - GLOBAL CLIMATE CHANGE INTELLIGENCE
# REAL NASA GISTEMP DATA | GOOGLE COLAB | ONE CELL
# ============================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "-q", "install",
    "pandas", "numpy", "plotly", "requests"
])

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import requests
import zipfile
from pathlib import Path
from io import StringIO
from plotly.io import to_html
from IPython.display import display

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

PROJECT = Path("/content/Global_Climate_Change_Intelligence")
PROJECT.mkdir(parents=True, exist_ok=True)

DOWNLOAD_ZIP = False

SOURCE_URL = (
    "https://data.giss.nasa.gov/gistemp/"
    "tabledata_v4/GLB.Ts+dSST.csv"
)

RAW_FILE = PROJECT / "nasa_original_data.csv"

# ------------------------------------------------------------
# 2. DOWNLOAD REAL NASA CLIMATE DATA
# ------------------------------------------------------------

print("Downloading real NASA GISTEMP climate data...")

if not RAW_FILE.exists() or RAW_FILE.stat().st_size == 0:

    response = requests.get(
        SOURCE_URL,
        timeout=90,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    response.raise_for_status()

    RAW_FILE.write_bytes(response.content)

else:
    print("Using previously downloaded NASA dataset.")

print("Dataset file ready!")

# ------------------------------------------------------------
# 3. READ AND IDENTIFY TABLE HEADER
# ------------------------------------------------------------

content = RAW_FILE.read_text(
    encoding="utf-8-sig",
    errors="replace"
)

lines = content.splitlines()

header_index = None

for i, line in enumerate(lines):
    cells = [
        x.strip().strip('"')
        for x in line.split(",")
    ]

    if (
        len(cells) >= 13
        and cells[0].lower() == "year"
        and cells[1].lower() == "jan"
        and cells[12].lower() == "dec"
    ):
        header_index = i
        break

if header_index is None:
    raise ValueError(
        "NASA CSV header not recognised. "
        "Source format may have changed."
    )

raw = pd.read_csv(
    StringIO("\n".join(lines[header_index:])),
    dtype=str,
    on_bad_lines="skip"
)

raw.columns = [
    str(c).strip()
    for c in raw.columns
]

print("Original columns:", list(raw.columns))

# ------------------------------------------------------------
# 4. CLEAN DATA
# ------------------------------------------------------------

months = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

required = ["Year"] + months

missing = [
    c for c in required
    if c not in raw.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )

df = raw[required].copy()

df["Year"] = pd.to_numeric(
    df["Year"], errors="coerce"
)

for col in months:

    df[col] = pd.to_numeric(
        df[col].astype(str).str.strip(),
        errors="coerce"
    )

    # NASA uses *** for unavailable values.
    # Missing values stay missing.
    df.loc[
        ~df[col].between(-100, 100),
        col
    ] = np.nan

df = df.dropna(subset=["Year"])
df["Year"] = df["Year"].astype(int)

df = df.drop_duplicates(
    subset=["Year"],
    keep="first"
)

df = df.sort_values("Year").reset_index(drop=True)

# NASA anomalies are reported in hundredths of a degree C.
# Convert to degrees Celsius.
for col in months:
    df[col] = df[col] / 100.0

monthly = df.melt(
    id_vars="Year",
    value_vars=months,
    var_name="Month",
    value_name="Temperature_Anomaly_C"
)

monthly["Month_Number"] = monthly["Month"].map({
    month: i + 1
    for i, month in enumerate(months)
})

monthly["Date"] = pd.to_datetime(
    dict(
        year=monthly["Year"],
        month=monthly["Month_Number"],
        day=1
    ),
    errors="coerce"
)

monthly = monthly.dropna(
    subset=["Temperature_Anomaly_C", "Date"]
)

monthly = monthly.sort_values(
    "Date"
).reset_index(drop=True)

if monthly.empty:
    raise ValueError("No valid climate observations.")

# Annual mean from available monthly anomalies
annual = (
    monthly.groupby("Year", as_index=False)
    .agg(
        Annual_Anomaly_C=(
            "Temperature_Anomaly_C", "mean"
        ),
        Available_Months=(
            "Temperature_Anomaly_C", "count"
        )
    )
)

# Use complete years for annual trend comparisons
annual_complete = annual[
    annual["Available_Months"] == 12
].copy()

if annual_complete.empty:
    raise ValueError("No complete annual records found.")

annual_complete["Rolling_5Y_C"] = (
    annual_complete["Annual_Anomaly_C"]
    .rolling(5, min_periods=5)
    .mean()
)

annual_complete["Rolling_10Y_C"] = (
    annual_complete["Annual_Anomaly_C"]
    .rolling(10, min_periods=10)
    .mean()
)

annual_complete["Decade"] = (
    annual_complete["Year"] // 10 * 10
)

monthly.to_csv(
    PROJECT / "cleaned_monthly_climate_data.csv",
    index=False
)

annual_complete.to_csv(
    PROJECT / "cleaned_annual_climate_data.csv",
    index=False
)

print("\nDATA CLEANING COMPLETED")
print("Monthly observations:", len(monthly))
print("Complete years:", len(annual_complete))

display(monthly.head())

# ------------------------------------------------------------
# 5. KPI SUMMARY
# ------------------------------------------------------------

warmest = annual_complete.loc[
    annual_complete["Annual_Anomaly_C"].idxmax()
]

coldest = annual_complete.loc[
    annual_complete["Annual_Anomaly_C"].idxmin()
]

kpis = {
    "Monthly Observations": len(monthly),
    "Complete Years": len(annual_complete),
    "First Complete Year": int(
        annual_complete["Year"].min()
    ),
    "Last Complete Year": int(
        annual_complete["Year"].max()
    ),
    "Warmest Year": int(warmest["Year"]),
    "Warmest Annual Anomaly C": round(
        warmest["Annual_Anomaly_C"], 3
    ),
    "Coldest Year": int(coldest["Year"]),
    "Coldest Annual Anomaly C": round(
        coldest["Annual_Anomaly_C"], 3
    )
}

kpi_df = pd.DataFrame(
    list(kpis.items()),
    columns=["KPI", "Value"]
)

kpi_df.to_csv(
    PROJECT / "kpi_summary.csv",
    index=False
)

print("\nGLOBAL CLIMATE KPIs")
display(kpi_df)

# ------------------------------------------------------------
# 6. INTERACTIVE PLOTLY CHARTS
# ------------------------------------------------------------

figures = []

def show_chart(fig, title):

    fig.update_layout(
        title=title,
        template="plotly_white",
        height=500,
        margin=dict(
            l=45, r=40, t=80, b=55
        )
    )

    figures.append((title, fig))
    fig.show()

# CHART 1 - ANNUAL GLOBAL TEMPERATURE ANOMALY

show_chart(
    px.line(
        annual_complete,
        x="Year",
        y="Annual_Anomaly_C",
        markers=True
    ),
    "1. Global Annual Temperature Anomaly"
)

# CHART 2 - ANNUAL ANOMALY BAR CHART

show_chart(
    px.bar(
        annual_complete,
        x="Year",
        y="Annual_Anomaly_C",
        color="Annual_Anomaly_C",
        color_continuous_scale="RdBu_r"
    ),
    "2. Annual Temperature Anomalies by Year"
)

# CHART 3 - 5-YEAR ROLLING TREND

show_chart(
    px.line(
        annual_complete,
        x="Year",
        y="Rolling_5Y_C"
    ),
    "3. Five-Year Rolling Temperature Trend"
)

# CHART 4 - 10-YEAR ROLLING TREND

show_chart(
    px.line(
        annual_complete,
        x="Year",
        y="Rolling_10Y_C"
    ),
    "4. Ten-Year Rolling Temperature Trend"
)

# CHART 5 - DECADAL AVERAGE

decadal = (
    annual_complete.groupby(
        "Decade", as_index=False
    )["Annual_Anomaly_C"]
    .mean()
)

show_chart(
    px.bar(
        decadal,
        x="Decade",
        y="Annual_Anomaly_C",
        color="Annual_Anomaly_C",
        color_continuous_scale="RdBu_r"
    ),
    "5. Average Temperature Anomaly by Decade"
)

# CHART 6 - MONTHLY GLOBAL TREND

show_chart(
    px.line(
        monthly,
        x="Date",
        y="Temperature_Anomaly_C"
    ),
    "6. Monthly Global Temperature Anomaly"
)

# CHART 7 - MONTH VS YEAR HEATMAP

heat = monthly.pivot_table(
    index="Year",
    columns="Month",
    values="Temperature_Anomaly_C",
    aggfunc="mean"
)

heat = heat.reindex(columns=months)

show_chart(
    px.imshow(
        heat,
        aspect="auto",
        color_continuous_scale="RdBu_r",
        color_continuous_midpoint=0,
        labels={
            "x": "Month",
            "y": "Year",
            "color": "Anomaly C"
        }
    ),
    "7. Monthly Temperature Anomaly Heatmap"
)

# CHART 8 - WARMEST YEARS

top_years = annual_complete.nlargest(
    15, "Annual_Anomaly_C"
).sort_values("Annual_Anomaly_C")

show_chart(
    px.bar(
        top_years,
        x="Annual_Anomaly_C",
        y=top_years["Year"].astype(str),
        orientation="h",
        color="Annual_Anomaly_C",
        labels={"y": "Year"}
    ),
    "8. Top 15 Warmest Years"
)

# ------------------------------------------------------------
# 7. EXPORT CSV REPORTS
# ------------------------------------------------------------

decadal.to_csv(
    PROJECT / "summary_decadal_temperature.csv",
    index=False
)

month_summary = (
    monthly.groupby("Month", as_index=False)
    .agg(
        Average_Anomaly_C=(
            "Temperature_Anomaly_C", "mean"
        ),
        Observations=(
            "Temperature_Anomaly_C", "count"
        )
    )
)

month_summary["Month"] = pd.Categorical(
    month_summary["Month"],
    categories=months,
    ordered=True
)

month_summary = month_summary.sort_values("Month")

month_summary.to_csv(
    PROJECT / "summary_by_month.csv",
    index=False
)

annual_complete.nlargest(
    15, "Annual_Anomaly_C"
).to_csv(
    PROJECT / "top_15_warmest_years.csv",
    index=False
)

# ------------------------------------------------------------
# 8. COMBINED INTERACTIVE DASHBOARD
# ------------------------------------------------------------

html = """
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Global Climate Change Intelligence</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f4f6f9;
    margin: 25px;
}
h1, p {
    text-align: center;
}
.chart {
    background: white;
    border-radius: 12px;
    padding: 15px;
    margin: 20px auto;
    max-width: 1300px;
}
</style>
</head>
<body>
<h1>GLOBAL CLIMATE CHANGE INTELLIGENCE</h1>
<p>NASA GISTEMP Global Temperature Anomalies</p>
"""

for i, (title, fig) in enumerate(figures):

    html += (
        '<div class="chart">'
        + to_html(
            fig,
            full_html=False,
            include_plotlyjs=(i == 0)
        )
        + "</div>"
    )

html += "</body></html>"

dashboard_path = (
    PROJECT / "Global_Climate_Dashboard.html"
)

dashboard_path.write_text(
    html,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 9. README
# ------------------------------------------------------------

readme = f"""
PROJECT 5 - GLOBAL CLIMATE CHANGE INTELLIGENCE

Source:
NASA GISS Surface Temperature Analysis
{SOURCE_URL}

Monthly observations: {len(monthly)}
Complete years: {len(annual_complete)}
Interactive charts: {len(figures)}

Temperature values represent anomalies
relative to NASA's 1951-1980 baseline.

Original NASA monthly table values are
hundredths of a degree Celsius.
Converted to degrees Celsius for analysis.

No synthetic climate records generated.

Limitations:
This is global temperature anomaly data.
It does not include CO2 emissions,
rainfall, sea levels or regional weather.

Partial years are excluded from annual
comparisons but retained in monthly data.
"""

(PROJECT / "README.txt").write_text(
    readme,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 10. CREATE ZIP
# ------------------------------------------------------------

zip_path = Path(
    "/content/Global_Climate_Change_Intelligence.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:

    for file in PROJECT.rglob("*"):

        if file.is_file():
            archive.write(
                file,
                arcname=file.relative_to(
                    PROJECT.parent
                )
            )

# ------------------------------------------------------------
# 11. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 45)
print("PROJECT 5 COMPLETED")
print("=" * 45)

print("Source: NASA GISTEMP")
print("Monthly records:", len(monthly))
print("Complete years:", len(annual_complete))
print("Interactive charts:", len(figures))
print("Dashboard:", dashboard_path)
print("ZIP:", zip_path)
print("Automatic download: OFF")

if DOWNLOAD_ZIP:

    from google.colab import files
    files.download(str(zip_path))
